# Distill suite: NMSE val и косинусы (TensorBoard)

Палитра как в `analyze_all_teacher_pair_metrics_vs_tvd_binned.py` (`group_colors`). Кривые: `nmse/val` в логарифмической шкале по оси Y. Столбцы на последнем step: выравнивание Transcoder (key/value) и SAE (decoder/encoder vs GT).

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tensorboard.backend.event_processing import event_accumulator as ea

_cwd = Path.cwd().resolve()
# run from the repo root or from case_study_1/
TOY_ROOT = _cwd if (_cwd / "out").is_dir() else _cwd.parent

DISTILL_SUITE = TOY_ROOT / "out" / "distill_suite"

# Как в analyze_all_teacher_pair_metrics_vs_tvd_binned.py (group_colors)
PALETTE = {
    "dark": "#7d1111",
    "mid": "#85472a",
    "light": "#dca35b",
}

# Bar chart (alignment): цвета и подпись оси — рядом с PALETTE, чтобы ячейка с barh работала без ячейки со стилем
BAR_COLORS_ALIGNMENT = [
    PALETTE["dark"],  # Transcoder (keys)
    "#b05050",  # Transcoder (values)
    PALETTE["light"],  # SAE (decoder)
    "#e8c686",  # SAE (encoder)
]
BAR_COLOR_RANDOM_BASELINE = "#9a9a9a"
COS_SIM_AXIS_LABEL = "Mean-Max Cosine Similarity"

RUN_IDS = {
    "mlp_h8192": "h8192_k8_relu1",
    "sae_h8192": "sae_h8192_k8_relu1_pb0",
    "mlp_h1024": "h1024_knone_relu1",
}

RUNS: dict[str, dict] = {
    "h8192_k8_relu1": {
        "dir": DISTILL_SUITE / "target_models/results_match_real/h8192_k8_relu1",
        "label": "Transcoder",
        "color": PALETTE["dark"],
    },
    "sae_h8192_k8_relu1_pb0": {
        "dir": DISTILL_SUITE
        / "target_models/results_match_real/sae_h8192_k8_relu1_pb0",
        "label": "SAE",
        "color": PALETTE["mid"],
    },
    "h1024_knone_relu1": {
        "dir": DISTILL_SUITE
        / "target_models/results_real_h10240_knone/h1024_knone_relu1",
        "label": "Dense studentMLP",
        "color": PALETTE["light"],
    },
}

ROLLING_WINDOW = 11

for key, cfg in RUNS.items():
    d = cfg["dir"]
    if not d.is_dir():
        print(f"WARN: нет каталога {d}")
    else:
        ev = list(d.glob("events.out.tfevents*"))
        print(f"OK {key}: {len(ev)} event file(s)")

In [ ]:
def load_scalars(run_dir: Path) -> dict[str, pd.DataFrame]:
    acc = ea.EventAccumulator(str(run_dir), size_guidance={ea.SCALARS: 0})
    acc.Reload()
    tags = acc.Tags().get("scalars", [])
    out: dict[str, pd.DataFrame] = {}
    for tag in tags:
        events = acc.Scalars(tag)
        if not events:
            continue
        steps = np.array([e.step for e in events], dtype=np.int64)
        vals = np.array([e.value for e in events], dtype=np.float64)
        out[tag] = pd.DataFrame({"step": steps, "value": vals})
    return out


def smooth_series(y: np.ndarray, window: int) -> np.ndarray:
    if window <= 1:
        return y
    w = min(window, len(y))
    if w % 2 == 0:
        w -= 1
    if w < 3:
        return y
    s = pd.Series(y)
    return s.rolling(window=w, center=True, min_periods=1).mean().to_numpy()


DATA: dict[str, dict[str, pd.DataFrame]] = {}
for run_id, cfg in RUNS.items():
    DATA[run_id] = load_scalars(cfg["dir"])

TAG_NMSE_VAL = "nmse/val"

In [ ]:
# Стиль как в analyze_all_teacher_pair_metrics_vs_tvd_binned.py: grid alpha=0.25
mpl.rcParams.update(
    {
        "figure.dpi": 120,
        "savefig.dpi": 300,
        "font.size": 14,
        "axes.labelsize": 12,
        "legend.fontsize": 10,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "axes.linewidth": 0.9,
        "lines.linewidth": 1.7,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "grid.linewidth": 0.6,
    }
)

FIG_SQ = (7, 5.2)


def plot_nmse_val_log10(
    ax: plt.Axes,
    run_ids: list[str],
    alpha_raw: float = 0.45,
    legend_loc: str = "upper right",  # Изменено: теперь правый верхний угол
):
    """NMSE val; ось Y — log10; верх оси 10^0; в легенде — итоговый NMSE в скобках."""
    for run_id in run_ids:
        cfg = RUNS[run_id]
        df = DATA[run_id][TAG_NMSE_VAL]
        x = df["step"].to_numpy()
        y = np.maximum(df["value"].to_numpy(), 1e-30)
        ys = np.maximum(smooth_series(y, ROLLING_WINDOW), 1e-30)
        fin = float(ys[-1])
        label = f'{cfg["label"]} (final NMSE = {fin:.4f})'  # Изменено: явно указано "final NMSE ="
        color = cfg["color"]
        ax.plot(x, y, color=color, alpha=alpha_raw, linewidth=1.0)
        ax.plot(x, ys, color=color, label=label, linewidth=1.85)
    ax.set_xlabel("Training step")
    ax.set_ylabel("Normalized Mean Squared Error (NMSE)")
    ax.set_yscale("log")
    lo, _hi = ax.get_ylim()
    ax.set_ylim(lo, 10**0)
    ax.legend(loc=legend_loc, frameon=False)


def decorate_nmse_three_run_plot(ax: plt.Axes, run_ids: list[str]) -> None:
    """Стрелка SAE→Transcoder с NMSE_SAE/NMSE_Transcoder (итоговые NMSE — в легенде plot_nmse_val_log10)."""
    if len(run_ids) != 3:
        return
    finals: list[float] = []
    for run_id in run_ids:
        df = DATA[run_id][TAG_NMSE_VAL]
        y = np.maximum(df["value"].to_numpy(), 1e-30)
        ys = np.maximum(smooth_series(y, ROLLING_WINDOW), 1e-30)
        finals.append(float(ys[-1]))

    v_t, v_s = finals[0], finals[1]
    if v_t <= 1e-30 or v_s <= 1e-30:
        return
    ratio = v_s / v_t
    xlim = ax.get_xlim()
    x_arrow = xlim[0] + 0.88 * (xlim[1] - xlim[0])
    ax.annotate(
        "",
        xy=(x_arrow, v_t),
        xytext=(x_arrow, v_s),
        arrowprops=dict(
            arrowstyle="-|>",
            color="#222222",
            lw=2,
            shrinkA=4,
            shrinkB=4,
        ),
        zorder=10,
        clip_on=False,
    )
    y_mid = float(np.exp((np.log(v_s) + np.log(v_t)) / 2.0))
    ax.annotate(
        f"{ratio:.1f}×",
        xy=(x_arrow, y_mid),
        xytext=(7, 0),
        textcoords="offset points",
        ha="left",
        va="center",
        fontsize=13,
        color="#222222",
        zorder=11,
        clip_on=False,
    )

### 1) NMSE val (log), только Transcoder и SAE

In [ ]:
fig, ax = plt.subplots(figsize=FIG_SQ, constrained_layout=True)
plot_nmse_val_log10(
    ax,
    [RUN_IDS["mlp_h8192"], RUN_IDS["sae_h8192"]],
)
plt.show()

### 2) То же + Dense MLP

In [ ]:
fig, ax = plt.subplots(figsize=FIG_SQ, constrained_layout=True)
run_three = [
    RUN_IDS["mlp_h8192"],
    RUN_IDS["sae_h8192"],
    RUN_IDS["mlp_h1024"],
]
plot_nmse_val_log10(ax, run_three, legend_loc="upper right")
decorate_nmse_three_run_plot(ax, run_three)
plt.show()

### 3) Последний step: alignment (Transcoder keys/values, SAE decoder/encoder vs GT)

In [ ]:
# Стиль как в analyze_all_teacher_pair_metrics_vs_tvd_binned.py: grid alpha=0.25
mpl.rcParams.update(
    {
        "figure.dpi": 120,
        "savefig.dpi": 300,
        "font.size": 11,
        "axes.labelsize": 12,
        "legend.fontsize": 10,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "axes.linewidth": 0.9,
        "lines.linewidth": 1.7,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "grid.linewidth": 0.6,
    }
)

FIG_SQ = (5.2, 5.2)


def plot_nmse_val_log10(
    ax: plt.Axes,
    run_ids: list[str],
    alpha_raw: float = 0.45,
    legend_loc: str = "upper right",  # Изменено: теперь правый верхний угол
):
    """NMSE val; ось Y — log10; верх оси 10^0; в легенде — итоговый NMSE в скобках."""
    for run_id in run_ids:
        cfg = RUNS[run_id]
        df = DATA[run_id][TAG_NMSE_VAL]
        x = df["step"].to_numpy()
        y = np.maximum(df["value"].to_numpy(), 1e-30)
        ys = np.maximum(smooth_series(y, ROLLING_WINDOW), 1e-30)
        fin = float(ys[-1])
        label = f'{cfg["label"]} (final NMSE = {fin:.4f})'  # Изменено: явно указано "final NMSE ="
        color = cfg["color"]
        ax.plot(x, y, color=color, alpha=alpha_raw, linewidth=1.0)
        ax.plot(x, ys, color=color, label=label, linewidth=1.85)
    ax.set_xlabel("step")
    ax.set_ylabel("NMSE")
    ax.set_yscale("log")
    lo, _hi = ax.get_ylim()
    ax.set_ylim(lo, 10**0)
    ax.legend(loc=legend_loc, frameon=False)


def decorate_nmse_three_run_plot(ax: plt.Axes, run_ids: list[str]) -> None:
    """Стрелка SAE→Transcoder с NMSE_SAE/NMSE_Transcoder (итоговые NMSE — в легенде plot_nmse_val_log10)."""
    if len(run_ids) != 3:
        return
    finals: list[float] = []
    for run_id in run_ids:
        df = DATA[run_id][TAG_NMSE_VAL]
        y = np.maximum(df["value"].to_numpy(), 1e-30)
        ys = np.maximum(smooth_series(y, ROLLING_WINDOW), 1e-30)
        finals.append(float(ys[-1]))

    v_t, v_s = finals[0], finals[1]
    if v_t <= 1e-30 or v_s <= 1e-30:
        return
    ratio = v_s / v_t
    xlim = ax.get_xlim()
    x_arrow = xlim[0] + 0.88 * (xlim[1] - xlim[0])
    ax.annotate(
        "",
        xy=(x_arrow, v_t),
        xytext=(x_arrow, v_s),
        arrowprops=dict(
            arrowstyle="-|>",
            color="#222222",
            lw=1.55,
            shrinkA=4,
            shrinkB=4,
        ),
        zorder=10,
        clip_on=False,
    )
    y_mid = float(np.exp((np.log(v_s) + np.log(v_t)) / 2.0))
    ax.annotate(
        f"{ratio:.1f}×",
        xy=(x_arrow, y_mid),
        xytext=(11, 0),
        textcoords="offset points",
        ha="left",
        va="center",
        fontsize=10,
        color="#222222",
        zorder=11,
        clip_on=False,
    )

import pickle

import torch
import torch.nn.functional as F

from matplotlib.patches import FancyArrowPatch

rid_mlp = RUN_IDS["mlp_h8192"]
rid_sae = RUN_IDS["sae_h8192"]

TEACHER_META_PATH = (
    DISTILL_SUITE / "target_models/results_match_real/teacher_meta.pkl"
)


def last_scalar(run_id: str, tag: str) -> float:
    df = DATA[run_id][tag].sort_values("step")
    return float(df.iloc[-1]["value"])


def mean_max_cosine_sim(
    gt_rows: torch.Tensor, cand_rows: torch.Tensor, chunk_size: int = 1024
) -> float:
    """Как в distill_mlp._mean_max_cosine_sim: max cos по cand, затем mean по GT."""
    gt = F.normalize(gt_rows.float(), dim=1, eps=1e-12)
    cand = F.normalize(cand_rows.float(), dim=1, eps=1e-12)
    vals = []
    for i in range(0, gt.shape[0], chunk_size):
        g = gt[i : i + chunk_size]
        vals.append((g @ cand.t()).max(dim=1).values)
    return float(torch.cat(vals, dim=0).mean().item())


def random_gaussian_baseline(n_rows: int, n_cols: int, seed: int) -> float:
    """Random baseline: GT и candidate — i.i.d. Gaussian (как нулевая структура)."""
    gen = torch.Generator().manual_seed(seed)
    gt = torch.randn((n_rows, n_cols), generator=gen)
    cand = torch.randn((n_rows, n_cols), generator=gen)
    return mean_max_cosine_sim(gt, cand)


with open(TEACHER_META_PATH, "rb") as f:
    _teacher_meta = pickle.load(f)
_ckpt = torch.load(_teacher_meta["ckpt"], map_location="cpu")
_layer = int(_teacher_meta["layer_idx"])
_w1 = _ckpt["model"][f"blocks.{_layer}.mlp.fc1.weight"]
_w2 = _ckpt["model"][f"blocks.{_layer}.mlp.fc2.weight"]
_n_gt, _d_vec = _w1.shape

# Короткие подписи категорий + горизонтальные бары: текст читается слева, не по диагонали.
labels = [
    "Transcoder (keys)",
    "Transcoder (values)",
    "SAE (decoder)",
    "SAE (encoder)",
]
values = [
    last_scalar(rid_mlp, "params/key_max_cos_mean"),
    last_scalar(rid_mlp, "params/value_max_cos_mean"),
    last_scalar(rid_sae, "params/decoder_vs_gt_value_max_cos_mean"),
    last_scalar(rid_sae, "params/encoder_vs_gt_value_max_cos_mean"),
]
random_baseline = random_gaussian_baseline(_n_gt, _d_vec, seed=0)

labels = labels + ["Random Baseline"]
values = values + [random_baseline]
bar_colors = BAR_COLORS_ALIGNMENT + [BAR_COLOR_RANDOM_BASELINE]

fig, ax = plt.subplots(figsize=(7, 6), constrained_layout=True)
y = np.arange(len(labels))

for i, (label, val, color) in enumerate(zip(labels, values, bar_colors)):
    if label == "Random Baseline":
        bars = ax.barh(
            i,
            val,
            color=color,
            height=0.62,
            edgecolor="black",
            linewidth=0.8,
            alpha=0.9,
        )
    elif "Transcoder" in label:
        bars = ax.barh(
            i,
            val,
            color=color,
            height=0.62,
            edgecolor="black",
            linewidth=0.8,
            hatch="//",
            alpha=0.85,
        )
    else:
        bars = ax.barh(
            i,
            val,
            color=color,
            height=0.62,
            edgecolor="black",
            linewidth=0.8,
        )
    ax.bar_label(bars, labels=[f"{val:.4f}"], padding=4, fontsize=11)

ax.set_yticks(y, labels=labels)
ax.invert_yaxis()
ax.set_xlabel(COS_SIM_AXIS_LABEL, fontsize=13)
x_max = 0.8
ax.set_xlim(0.0, x_max)

# Стрелка между Transcoder (values) и SAE (decoder): показываем прирост по cos-sim
# i_tv, i_sd = 1, 2
# xv, xd = values[i_tv], values[i_sd]
# pct_vs_decoder = (xv / xd - 1.0) * 100.0 if xd > 1e-12 else float("nan")
# y_mid = (i_tv + i_sd) / 2.0
# arrow = FancyArrowPatch(
#     (xd, y_mid),
#     (xv, y_mid),
#     arrowstyle="|-|",
#     mutation_scale=9,
#     linewidth=1.35,
#     color="#222222",
#     zorder=5,
#     clip_on=False,
# )
# ax.add_patch(arrow)
# ax.annotate(
#     f"{pct_vs_decoder:+.1f}%",
#     xy=((xd + xv) / 2, y_mid + 0.1),
#     xytext=(0, 14),
#     textcoords="offset points",
#     ha="center",
#     va="bottom",
#     fontsize=13,
#     linespacing=1.15,
#     color="#222222",
#     zorder=6,
#     clip_on=False,
# )

plt.show()

### Экспорт

In [ ]:
OUT_DIR = TOY_ROOT / "figures_distill_suite_tb"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# fig, ax = plt.subplots(figsize=FIG_SQ, constrained_layout=True)
# plot_nmse_val_log10(ax, [RUN_IDS["mlp_h8192"], RUN_IDS["sae_h8192"]])
# fig.savefig(OUT_DIR / "nmse_val_log_mlp8192_sae.pdf", bbox_inches="tight")
# plt.close(fig)
# ... аналогично для остальных фигур
# print("Saved to", OUT_DIR)